# MicroDuck 一小时实验课

本 notebook 在 ROCm 容器里演示 UniLab + `microduck_rl_unilab` 的最小闭环：

1. 环境自检
2. 训练基础走路步态（`microduck_velocity_flat`）
3. 训练前向奔跑（`microduck_sprint_flat`）
4. 读取 TensorBoard 曲线并渲染视频

**时间盒：** 约 60 分钟。现场配置使用 `512` 个并行环境，每个阶段 `2000` 次 PPO iteration。

**预期：** 你能看到 reward 上升、episode 变长，以及“开始迈步/开始往前冲”的视频。
不会在这一小时内复现发表级的 1.68 m/s 或长时直线 checkpoint。


In [ ]:
from pathlib import Path
import sys

LAB_ROOT = Path("/workspace/microduck_rl_lab")
sys.path.insert(0, str(LAB_ROOT / "scripts"))

from lab_utils import (
    TrainJob,
    check_environment,
    latest_run_dir,
    plot_training_curves,
    render_playback,
    run_train,
    show_video,
    microduck_root,
)

check_environment()


## 0. 冒烟：2 iteration

先确认 registry、MuJoCo 资产和 ROCm 设备都能跑通，再进入长一点的训练。


In [ ]:
smoke = TrainJob(
    task="microduck_velocity_flat",
    task_name="MicroduckVelocityFlat",
    num_envs=4,
    max_iterations=2,
    save_interval=2,
    run_name="smoke",
)
run_train(smoke)
print("smoke ok")


## 1. 基础步态：velocity owner

`microduck_velocity_flat` 是上游 velocity 任务的 Manager-Based owner。
这里训练约 2000 iteration，观察 mean reward / episode length 是否抬升。


In [ ]:
velocity_job = TrainJob(
    task="microduck_velocity_flat",
    task_name="MicroduckVelocityFlat",
    num_envs=512,
    max_iterations=2000,
    save_interval=200,
    run_name="velocity_hour",
)
run_train(velocity_job)
velocity_run = latest_run_dir(velocity_job)
velocity_run


In [ ]:
fig = plot_training_curves(velocity_run, title="Velocity hour lab")
fig


In [ ]:
velocity_video = render_playback(
    task="microduck_velocity_flat",
    run_dir=velocity_run,
    play_steps=500,
)
show_video(velocity_video)


## 2. 奔跑姿态：sprint owner

切换到 `microduck_sprint_flat`。奖励会更强调前向速度，头部姿态通常更差。
这是正常的——发表级 head-upright / straight-run 需要后续续训，不在本小时范围内。


In [ ]:
sprint_job = TrainJob(
    task="microduck_sprint_flat",
    task_name="MicroduckSprintFlat",
    num_envs=512,
    max_iterations=2000,
    run_name="sprint_hour",
)
run_train(sprint_job)
sprint_run = latest_run_dir(sprint_job)
sprint_run


In [ ]:
fig = plot_training_curves(sprint_run, title="Sprint hour lab")
fig


In [ ]:
sprint_video = render_playback(
    task="microduck_sprint_flat",
    run_dir=sprint_run,
    play_steps=500,
    speed=2.2,
)
show_video(sprint_video)


## 3. 发表级对照片（若 examples 已存在于任务仓）

如果 `microduck_rl_unilab` 镜像里已经带有 `examples/sprint_*`，可以直接播放
夜训后的 checkpoint，对比本小时短训与发表结果之间的差距。


In [ ]:
examples_root = microduck_root() / "examples"
for name in ("sprint_speed_1p68", "sprint_head_upright", "sprint_straight_long"):
    example_dir = examples_root / name
    gif = example_dir / "play_video_side.gif"
    mp4 = example_dir / "play_video.mp4"
    metrics = example_dir / "metrics.json"
    print(name, "dir=", example_dir.exists(), "gif=", gif.exists(), "mp4=", mp4.exists(), "metrics=", metrics.exists())

head_upright = examples_root / "sprint_head_upright"
if (head_upright / "play_video_side.gif").is_file():
    from IPython.display import Image, display
    display(Image(filename=str(head_upright / "play_video_side.gif")))
else:
    print("Published examples not baked into this MICRODUCK_REF; skip or rebuild with a newer ref.")


## 下一步

- 完整训练得失见任务仓 `tutorial` 分支的 `results/*.jsonl`
- 速度 / 头正 / 长时直线三个发表点见 `examples/sprint_*`
- 长时直线评测必须使用 `microduck_sprint_straightfix_flat` owner，而不是本 notebook 的 sprint flat owner
